Install Python Virtual Environment

In [1]:
!pip install uv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.6/20.6 MB 48.8 MB/s eta 0:00:00


Install dbt-core and dbt-adapter

In [2]:
!uv pip install --system dbt-core dbt-snowflake

Using Python 3.13.16 environment at: /usr
Resolved 79 packages in 1.21s
Prepared 29 packages in 5.86s
Uninstalled 2 packages in 3ms
Installed 29 packages in 88ms
 + agate==1.9.1
 + asn1crypto==1.5.1
 + boto3==1.43.108
 + botocore==1.43.108
 - certifi==2026.7.22
 + certifi==2025.1.31
 + colorama==0.4.6
 + daff==1.4.2
 + dbt-adapters==1.24.5
 + dbt-common==1.39.0
 + dbt-core==1.12.5
 + dbt-core-experimental-parser==2.0.5
 + dbt-extractor==0.6.0
 + dbt-protos==1.0.594
 + dbt-snowflake==1.12.1
 + deepdiff==8.6.2
 - filelock==4.0.7
 + filelock==3.32.7
 + isodate==0.7.2
 + jmespath==1.1.0
 + leather==0.4.1
 + mashumaro==3.17
 + metricflow==0.213.0
 + orderly-set==5.5.0
 + parsedatetime==2.6
 + pathspec==1.0.4
 + pytimeparse==1.1.9
 + rapidfuzz==3.14.6
 + s3transfer==0.19.2
 + snowflake-connector-python==4.8.0
 + snowplow-tracker==1.1.0


Check to see if dbt is installed

In [3]:
!dbt --version

Core:
  - installed: 1.12.5
  - latest:    1.12.5 - Up to date!

Plugins:
  - snowflake: 1.12.1 - Up to date!




Create the directory which will hold your profiles.yml file -> this is your 'connection string'

In [4]:
#set up the dbt directory then create your profile
!mkdir -p /root/.dbt/

In [52]:
#create database dbt_google_colab;
#use database dbt_google_colab;
#create schema bronze_level_schema;

In [53]:
!dbt debug #verify connection

19:04:48  Running with dbt=1.12.5
19:04:48  dbt version: 1.12.5
19:04:48  python version: 3.13.16
19:04:48  python path: /usr/bin/python3
19:04:48  os info: Linux-6.6.122+-x86_64-with-glibc2.39
19:04:48  Using profiles dir at /root/.dbt
19:04:48  Using profiles.yml file at /root/.dbt/profiles.yml
19:04:48  Using dbt_project.yml file at /content/dbt_parent_project/dbt_snowflake_colab/dbt_project.yml
19:04:48  Configuration:
19:04:48    profiles.yml file [ERROR invalid]
19:04:48    dbt_project.yml file [OK found and valid]
19:04:48  Required dependencies:
19:04:48   - git [OK found]

19:04:48  Connection test skipped since no profile was found
19:04:48  1 check failed:
19:04:48  Profile loading failed for the following reason:
Runtime Error
  Could not find profile named 'dbt_snowflake_colab'




Code needed to produce the profiles.yml file

In [5]:
#Alternative set up - use secrets to generate profile.yml
import os
import yaml
from google.colab import userdata

Your account, user id and password are in a secrets file and will be accessed when dbt runs

In [7]:
## Alternative set up - use secrets to generate profile.yml on the fly
# 1. Fetch credentials securely from Colab Secrets
sf_account = userdata.get('SF_ACCOUNT')
sf_user = userdata.get('SF_USER')
sf_password = userdata.get('SF_PW')

Create the profiles.yml file which is the 'connection string' to your platform, database and default schema

In [8]:
# 2. Define your profile structure
# my project name is dbt_snowflake_in_colab
profile_data = {
    "dbt_snowflake_in_colab": {
        "target": "dev",
        "outputs": {
            "dev": {
                "type": "snowflake",
                "account": sf_account,
                "user": sf_user,
                "password": sf_password,
                "role": "accountadmin",          # Non-sensitive config can be hardcoded here
                "warehouse": "compute_wh",
                "database": "dbt_google_colab",
                "schema": "bronze_level_schema",
                "threads": 4,
                "client_session_keep_alive": False
            }
        }
    }
}

The profiles.yml file will be stored in the /root/.dbt/ directory

In [9]:
os.makedirs('/root/.dbt/', exist_ok=True)
with open('/root/.dbt/profiles.yml', 'w') as f:
    yaml.dump(profile_data, f, default_flow_style=False)

Check the version of dbt that you have installed

In [10]:
#Now dbt is ready to use
!dbt --version

Core:
  - installed: 1.12.5
  - latest:    1.12.5 - Up to date!

Plugins:
  - snowflake: 1.12.1 - Up to date!




This will provide the user manual for dbt

In [11]:
!dbt

Usage: dbt [OPTIONS] [COMMAND] [ARGS]...

  An ELT tool for managing your SQL transformations and data models. For more
  documentation on these commands, visit: docs.getdbt.com

Options:
  --cache-selected-only / --no-cache-selected-only
                                  At start of run, populate relational cache
                                  only for schemas containing selected nodes,
                                  or for all schemas of interest.
  -d, --debug / --no-debug        Display debug logging during dbt execution.
                                  Useful for debugging and making bug reports.
  --defer / --no-defer            If set, resolve unselected nodes by
                                  deferring to the manifest within the --state
                                  directory.
  --defer-state DIRECTORY         Override the state directory for deferral
                                  only.
  --deprecated-favor-state TEXT   Internal flag for deprecating old env v

You have installed dbt and now it's time to set up a Project. Create a parent folder to hold your dbt project. In this case I call it dbt_parent_folder

In [17]:
os.makedirs('/content/dbt_parent_folder/', exist_ok=True)

In [18]:
#Navigate to your project folder
%cd  /content/dbt_parent_folder/

/content/dbt_parent_folder


Checking the contents. At this point the folder is empty

In [20]:
!ls

Add the dbt project to this working folder

In [21]:
!dbt init dbt_snowflake_in_colab --skip-profile-setup

12:11:32  Running with dbt=1.12.5
12:11:32  Your new dbt project "dbt_snowflake_in_colab" was created!
Initialized new project in /content/dbt_parent_folder/dbt_snowflake_in_colab/dbt_snowflake_in_colab

For more information on how to configure the profiles.yml file,
please consult the dbt documentation here:

  https://docs.getdbt.com/docs/configure-your-profile

One more thing:

Need help? Don't hesitate to reach out to us via GitHub issues or on Slack:

  https://community.getdbt.com/

Happy modeling!



Navigate to the project folder. In my case it's /content/dbt_parent_folder/dbt_snowflake_in_colab/

In [22]:
#Navigate to your project folder
%cd /content/dbt_parent_folder/dbt_snowflake_in_colab/
!dbt run

/content/dbt_parent_folder/dbt_snowflake_in_colab
12:12:33  Running with dbt=1.12.5
12:12:34  Registered adapter: snowflake=1.12.1
12:12:35  Unable to do partial parsing because saved manifest not found. Starting full parse.
12:12:37  Found 2 models, 4 data tests, 562 macros
12:12:37  
12:12:37  Concurrency: 4 threads (target='dev')
12:12:37  
12:12:41  1 of 2 START sql table model bronze_level_schema.my_first_dbt_model ............ [RUN]
12:12:44  1 of 2 OK created sql table model bronze_level_schema.my_first_dbt_model ....... [SUCCESS 2 in 2.79s]
12:12:44  2 of 2 START sql view model bronze_level_schema.my_second_dbt_model ............ [RUN]
12:12:45  2 of 2 OK created sql view model bronze_level_schema.my_second_dbt_model ....... [SUCCESS 1 in 1.65s]
12:12:47  
12:12:47  Finished running 1 table model, 1 view model in 0 hours 0 minutes and 10.08 seconds (10.08s).
12:12:47  
12:12:47  Completed successfully
12:12:47  
12:12:47  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTA

Log into your platform and go to the database and schema that you specified. You will see 2 models that were created there by dbt. One is a view and the other is a table. This just ensures that you have connectivity and that dbt is working.
Happy Modeling!